## A/B-testing

1. Создать подключение к базе данных с помощью sqlite3 библиотеки.

In [6]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/checking-logs.sqlite")

2. Создать два фрейма данных: test_results и control_results.

In [7]:
query_test = """
SELECT "after" AS time,
       AVG((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS avg_diff
FROM test AS t
JOIN deadlines AS d ON LOWER(t.labname) = LOWER(d.labs)
WHERE julianday(first_view_ts) > julianday(t.first_commit_ts)
AND t.labname != 'project1'
UNION ALL
SELECT "before" AS time,
       AVG((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS avg_diff
FROM test AS t
JOIN deadlines AS d ON LOWER(t.labname) = LOWER(d.labs)
WHERE julianday(first_view_ts) < julianday(t.first_commit_ts)
AND t.labname != 'project1'
"""

test_results = pd.read_sql(query_test, conn)

test_results

,time,avg_diff
0,after,-61.156438
1,before,-103.953310


In [ ]:
query_control = """
SELECT "after" AS time,
       AVG((julianday(c.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS avg_diff
FROM control AS c
JOIN deadlines AS d ON LOWER(c.labname) = LOWER(d.labs)
WHERE julianday(first_view_ts) > julianday(c.first_commit_ts)
AND c.labname != 'project1'
UNION ALL
SELECT "before" AS time,
       AVG((julianday(c.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS avg_diff
FROM control AS c
JOIN deadlines AS d ON LOWER(c.labname) = LOWER(d.labs)
WHERE julianday(first_view_ts) < julianday(c.first_commit_ts)
AND c.labname != 'project1'
"""

control_results = pd.read_sql(query_control, conn)

control_results

,time,avg_diff
0,after,-99.901295
1,before,-113.232196


3. Закрыть соединение.

In [9]:
conn.close()

4. Ответ на вопрос о подтверждении гипотезы.

In [10]:
delta_test = abs(test_results["avg_diff"].iloc[1]) - abs(test_results["avg_diff"].iloc[0])
delta_contol = abs(control_results["avg_diff"].iloc[1]) - abs(control_results["avg_diff"].iloc[0])

print(f"Разница во времени в тестовой группе: {delta_test}\n",
      f"Разница во времени в контрольной группе: {delta_contol}\n", sep='')

print(f"Разница до посещения и после в тестовой группе существенна. \nВ контрольной группе подобного мы не наблюдаем, соответственно гипотеза ПОДТВЕРЖДЕНА")

Разница во времени в тестовой группе: 42.79687207436655
Разница во времени в контрольной группе: 13.330901912455545

Разница до посещения и после в тестовой группе существенна. 
В контрольной группе подобного мы не наблюдаем, соответственно гипотеза ПОДТВЕРЖДЕНА
